# Spectrum-reconstructing VAE: C 1s peak fitting

This replaces the old parameter-only variational predictor with a model whose decoder predicts physical peak parameters and reconstructs the spectrum through the same renderer. It uses a convolutional encoder, a 32-dimensional sampled latent state, supervised peak losses, a clean-spectrum loss, KL warmup, and free bits.

**This notebook does not contain pre-trained results.** Running its training cell on a suitable machine will create a best checkpoint and a complete held-out test report. The original instruction for this workspace is **not to run training on this machine**, so the notebooks are prepared and verified without starting optimization here.

Run from the project directory. Required packages are in `environment.yml` (PyTorch, NumPy, SciPy, h5py, pandas, matplotlib, Jupyter). To generate the compact data in a separate process, run `python c1s_lmfit_dataset.py --cases 10000 --replicates 5 --scans-per-file 100`. The notebook waits for the generator's completed `manifest.json`; it never reads half-written shards. Each notebook uses that same dataset, splits, renderer, and scoring code in `c1s_peak_pipeline.py`.

## 1. Architecture and settings

At training time the encoder maps a noisy spectrum to a distribution over latent descriptions. A sample is decoded into peak and background parameters, then the renderer recreates the clean spectrum. At evaluation the latent mean gives a reproducible prediction; a separate cell samples it repeatedly to inspect ambiguity. Sample spread is exploratory, not a calibrated confidence interval.

All models predict independent parameters: peak existence, shape, center, model width, area, the relevant shape-specific value, and background. The renderer calculates clean spectra, height, and final FWHM. Losses check both physical parameters and the recreated clean curve. The initial loss weights and model sizes are starting settings to compare on validation data, not claims of global optimality.

In [ ]:
from pathlib import Path
import json
import torch
import pandas as pd
from IPython.display import Image, display
from c1s_peak_pipeline import (
    DATA_DIR, OUTPUT_ROOT, RunConfig, seed_everything, wait_for_dataset,
    load_dataset, make_loaders, build_model, C1sRenderer, audit_renderer,
    train_model, plot_learning_history, choose_presence_threshold,
    evaluate_model, compare_available_models,
)

if not Path("c1s_peak_pipeline.py").is_file():
    raise RuntimeError("Open this notebook from the spectraVAE project directory.")
MODEL_NAME = "vae"
config = RunConfig(
    model_name=MODEL_NAME,
    dataset_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_ROOT / MODEL_NAME),
    wait_hours=24,
    poll_seconds=30,
    seed=42,
    batch_size=64,
    epochs=100,
    learning_rate=3e-4,
    patience=18,
    resume=True,
)
seed_everything(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Model:", MODEL_NAME, "| Device:", device)
print("Dataset:", config.dataset_dir)
print("Reports and checkpoints:", config.output_dir)


## 2. Wait for complete dataset

This cell can be run while generation is still in progress. The manifest is the completion marker. If generation does not finish within 24 hours, it gives a clear timeout rather than training on incomplete files. Repeated scans from the same synthetic case stay in one split.

In [ ]:
# The generator publishes manifest.json only after every NeXus shard is finished.
# This cell waits, verifies the file list, and loads the same case-separated splits
# for all three notebooks. It does not train a model.
manifest, manifest_sha = wait_for_dataset(
    config.dataset_dir, hours=config.wait_hours, poll_seconds=config.poll_seconds
)
data = load_dataset(config.dataset_dir, manifest)
loaders = make_loaders(data, config)
print("Manifest SHA256:", manifest_sha)
print("Scans:", {split: len(loader.dataset) for split, loader in loaders.items()})
print("Distinct cases:", {split: len(set(data["case_id"][data["split"] == split]))
                         for split in ("train", "validation", "test")})
print("Energy points:", len(data["energy"]), "| Shards:", len(manifest["shards"]))


## 3. Verify renderer before any training

The ground-truth peak and background parameters must reproduce the saved clean spectrum to within the documented numerical tolerance. If this check fails, the notebook stops before optimizing a model.

In [ ]:
model = build_model(MODEL_NAME).to(device)
renderer = C1sRenderer(data["energy"]).to(device)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
renderer_audit = audit_renderer(loaders["validation"], renderer, device, max_batches=2)
print("Ground-truth parameter → clean-spectrum renderer check:")
display(pd.DataFrame([renderer_audit]).T.rename(columns={0: "normalized error"}))


## 4. Train and retain the best model

The model learns peak presence, profile, center, width, area, applicable shape parameters, background, and clean-spectrum reconstruction together. The loss column plots help identify a task that stops improving or dominates the validation score. Training can resume from `latest.pt`; `best.pt` is the checkpoint used for reporting.

In [ ]:
# Training starts ONLY when this cell is run. Run this notebook on your training machine.
# The best checkpoint is selected on validation loss, saved as best.pt, and
# latest.pt allows resuming an interrupted run on the same dataset.
history, best_checkpoint = train_model(
    model, renderer, loaders, data, config, manifest_sha, device
)
print("Best epoch:", best_checkpoint["epoch"] + 1)
print("Best validation loss:", best_checkpoint["validation_total"])
display(history.tail(10).round(4))
display(Image(filename=str(plot_learning_history(history, config.output_dir))))


## 5. Fix the detection threshold using validation cases

The threshold controls which peak predictions count as present. Choosing it on validation avoids tuning to the held-out test data.

In [ ]:
# Use validation data to choose the peak-presence threshold.
# The held-out test split is not used for this decision.
threshold, threshold_table = choose_presence_threshold(
    model, loaders["validation"], device
)
threshold_table.to_csv(Path(config.output_dir) / "validation_thresholds.csv", index=False)
print("Chosen validation threshold:", threshold)
display(threshold_table.sort_values("threshold").round(3))


## 6. Evaluate held-out test cases

`test_true_peaks.csv` gives one row per actual peak, including missed peaks and separate errors for shape, center, model width, rendered FWHM, rendered height, area, and shape-specific parameters. `test_predicted_peaks.csv` identifies extra peaks. `test_scans.csv` includes full-spectrum and background errors. All rows include the source NeXus file and scan row. Metrics marked **all true** count a missed peak as a failure; median numeric errors are reported on detected peaks.

In [ ]:
# One held-out test evaluation. This writes full per-scan and per-peak CSVs,
# summary JSON, subgroup breakdown, shape confusion, case ranking, and figures.
test_summary, test_scans, test_peaks, test_cases = evaluate_model(
    model, renderer, loaders["test"], data, config, threshold, device,
    split="test", manifest_sha=manifest_sha
)
display(pd.DataFrame.from_dict(test_summary, orient="index", columns=["Test value"]).round(4))


## 7. Visual review

The first figure compares tasks. The subgroup chart and CSV show which chemical families, peak counts, weak peaks, and overlaps are difficult. The confusion chart shows shape mix-ups and misses. The gallery shows three easiest and three hardest held-out **cases**, ranked using repeated scans from each case; individual displayed scans are representative extremes within those cases. Solid colored curves are model peaks; dashed blue curves are the true synthetic components.

In [ ]:
report_dir = Path(config.output_dir)
for filename in (
    "task_overview.png", "failure_subgroups.png", "shape_confusion.png",
    "best_worst_cases.png",
):
    print(filename)
    display(Image(filename=str(report_dir / filename)))


In [ ]:
print("Best and worst held-out cases:")
display(pd.read_csv(report_dir / "best_worst_cases.csv").round(4))
print("Subgroups with at least 20 true peaks, ordered by area accuracy:")
breakdown = pd.read_csv(report_dir / "test_breakdown.csv")
display(breakdown[(breakdown["level"] == "peak") & (breakdown["n"] >= 20)]
        .sort_values("area_within_20_percent_all_true")
        [["group", "value", "n", "presence_recall", "shape_accuracy_all_true",
          "center_within_0p2_ev_all_true", "fwhm_within_0p2_ev_all_true",
          "height_within_20_percent_all_true", "area_within_20_percent_all_true"]]
        .head(20).round(3))
print("Largest individual peak-area errors among detected peaks:")
display(test_peaks[test_peaks["detected"]]
        .sort_values("area_relative_error", ascending=False)
        [["case_id", "source_file", "source_row", "component", "true_shape",
          "true_area_counts_ev", "predicted_area_counts_ev", "area_relative_error",
          "center_error_ev", "nearest_peak_ev", "area_fraction"]]
        .head(20).round(3))


## 8. Inspect latent-sampling spread

A VAE can produce several plausible parameter sets from one input. This samples 16 times on 200 held-out scans and compares sample-to-sample spread with actual center and area errors. A large spread may flag ambiguity, but these plots do **not** establish calibrated confidence intervals.

In [ ]:
from c1s_peak_pipeline import vae_uncertainty_report
uncertainty = vae_uncertainty_report(model, data, config, device, n_scans=200, draws=16)
display(uncertainty.head(20).round(4))
display(Image(filename=str(report_dir / "test_latent_sampling.png")))


## Compare completed runs

The comparison table updates as each of the three notebooks finishes. It uses exactly the same test cases and the same score definitions. Any differences are model results, not changes in splitting or evaluation. Synthetic test success is not yet evidence of accuracy on measured XPS spectra.

In [ ]:
comparison = compare_available_models()
print("Models with completed test reports:")
display(comparison.round(4))
print("Comparison CSV:", OUTPUT_ROOT / "model_comparison.csv")

if len(comparison):
    display(Image(filename=str(OUTPUT_ROOT / "model_comparison.png")))


## How to decide what to improve next

- **Low peak recall, especially under 5% area:** add more weak-peak examples or adjust the detection objective and threshold.
- **Good curve fit but wrong shape or peak count:** the decomposition is ambiguous; inspect the confusion table and individual component curves rather than relying on total-spectrum error.
- **Good detection but poor centers or widths:** review overlap groups and the center/width loss scales.
- **Good parameters but poor clean-spectrum fit:** inspect background accuracy and the renderer audit.
- **Large training–validation gap:** test broader generator variations and measured spectra before increasing model size.

The three notebooks report these cases separately; none can prove universal performance without external measured datasets.